<a href="https://colab.research.google.com/github/eduwerneck/projetos/blob/main/FloraCloud_pipeline_v03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌿 FloraCloud — Vegetation Mapping System

**Versão:** 0.3  
**Autores:** Eduardo Augusto Werneck Ribeiro  
**Instituição:** Instituto Federal Catarinense — Campus Brusque  
**Repositório:** https://github.com/eduwerneck/projetos  

---

## Método

Pipeline integrado para mapeamento 3D de vegetação com smartphone:

1. **Calibração radiométrica** — painel de reflectância (3 registros: entrada, percurso, saída)
2. **Structure from Motion** — reconstrução esparsa via pycolmap
3. **Nuvem densa alinhada** — Depth Anything V2 + scale alignment por pontos SfM
4. **VARI** — Visible Atmospherically Resistant Index por ponto (Gitelson et al., 2002)
5. **Exportação** — nuvem .ply + relatório JSON + visualizações

**Fórmula VARI:** `(G - R) / (G + R - B)`

---

## Protocolo de campo

1. Fotografar painel de calibração na **entrada** da parcela
2. Percurso combinado: perímetro + transectos internos (parcelas 30×30m)
3. Painel numa estaca central — visível em ~10% das fotos
4. Fotografar painel na **saída** da parcela
5. Mínimo recomendado: **50 fotos**, dia nublado, HDR desligado

---

## Requisitos de hardware

- **Recomendado:** GPU NVIDIA A100 ou similar (Google Colab Pro)
- **Mínimo:** GPU T4 (Google Colab gratuito) — mais lento

> ⚠️ Ative a GPU antes de rodar: `Ambiente de execução → Alterar tipo → A100`

## Célula 1 — Instalação

In [ ]:
# Instalar dependências
!pip install pycolmap open3d timm -q

import torch
import pycolmap
import open3d as o3d
import numpy as np

print(f'pycolmap: {pycolmap.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')
print('✅ Instalação concluída')

## Célula 2 — Metadados da sessão

Preencha antes de iniciar a coleta.

In [ ]:
from pathlib import Path
import os

# ─────────────────────────────────────────────
# PREENCHA AQUI
# ─────────────────────────────────────────────
METADATA = {
    'parcela':               'P-01',
    'operador':              'Eduardo',
    'data':                  '2026-03-26',
    'hora':                  '09:00',
    'gps_modo':              'celular',       # 'celular' | 'L1L2' | 'manual'
    'estaca_NW':             (-26.9184, -49.0612),
    'estaca_NE':             (-26.9181, -49.0608),
    'estaca_SW':             (-26.9188, -49.0612),
    'estaca_SE':             (-26.9185, -49.0608),
    'altura_min_estimada_m': 0.5,
    'altura_med_estimada_m': 2.0,
    'altura_max_estimada_m': 5.0,
    'fitofisionomia':        'subosque de floresta ombrófila densa',
    'luminosidade':          'nublado',       # 'ensolarado' | 'nublado' | 'parcial'
    'painel_batch':          '4001',
    'painel_red_ref':        19.7,            # % impresso no painel
    'painel_green_ref':      18.9,
}

# Criar estrutura de diretórios
DIRS = {
    'painel_entrada': Path('fotos/painel_entrada'),
    'percurso':       Path('fotos/percurso'),
    'painel_saida':   Path('fotos/painel_saida'),
    'colmap_input':   Path('colmap/input'),
    'colmap_output':  Path('colmap/output'),
    'resultados':     Path('resultados'),
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print(f'📋 Parcela: {METADATA["parcela"]} — {METADATA["data"]}')
print(f'🌿 {METADATA["fitofisionomia"]}')
print(f'☁️  Luminosidade: {METADATA["luminosidade"]}')
if METADATA['luminosidade'] == 'ensolarado':
    print('⚠️  ATENÇÃO: Coleta em dia ensolarado — sun flecks podem gerar inconsistência radiométrica.')
print('✅ Ambiente configurado')

## Célula 3 — Upload das fotos

Faça upload em 3 etapas: painel de entrada → fotos do percurso → painel de saída.

In [ ]:
from google.colab import files

# 3a — Painel de entrada
print('📷 Upload: painel na ENTRADA da parcela')
uploaded = files.upload()
for nome, dados in uploaded.items():
    with open(DIRS['painel_entrada'] / nome, 'wb') as f: f.write(dados)
print(f'✅ {len(uploaded)} foto(s) salva(s)')

In [ ]:
# 3b — Fotos do percurso
print('📷 Upload: fotos do PERCURSO (Ctrl+A para selecionar todas)')
uploaded = files.upload()
for nome, dados in uploaded.items():
    with open(DIRS['percurso'] / nome, 'wb') as f: f.write(dados)
    with open(DIRS['colmap_input'] / nome, 'wb') as f: f.write(dados)
n = len(list(DIRS['percurso'].glob('*')))
print(f'✅ {n} fotos salvas')
if n < 20:
    print('⚠️  Menos de 20 fotos — recomendado mínimo 50 para boa reconstrução')

In [ ]:
# 3c — Painel de saída
print('📷 Upload: painel na SAÍDA da parcela')
uploaded = files.upload()
for nome, dados in uploaded.items():
    with open(DIRS['painel_saida'] / nome, 'wb') as f: f.write(dados)
print(f'✅ {len(uploaded)} foto(s) salva(s)')

## Célula 4 — Calibração radiométrica

In [ ]:
import cv2
import numpy as np

def extrair_rgb_painel(caminho_foto):
    img = cv2.imread(str(caminho_foto))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    # ROI central onde está o painel cinza
    roi = img_rgb[int(h*0.35):int(h*0.65), int(w*0.30):int(w*0.70)]
    return np.mean(roi[:,:,0]), np.mean(roi[:,:,1]), np.mean(roi[:,:,2])

registros = {}
for momento, pasta in [('entrada', DIRS['painel_entrada']), ('saida', DIRS['painel_saida'])]:
    fotos = list(pasta.glob('*.jpg')) + list(pasta.glob('*.JPG')) + list(pasta.glob('*.jpeg'))
    if fotos:
        r, g, b = extrair_rgb_painel(fotos[0])
        fr = (METADATA['painel_red_ref']   / 100.0) / (r / 255.0)
        fg = (METADATA['painel_green_ref'] / 100.0) / (g / 255.0)
        registros[momento] = {'R': r, 'G': g, 'B': b, 'fator_R': fr, 'fator_G': fg}
        print(f'Painel {momento}: R={r:.1f} G={g:.1f} | fator_R={fr:.4f} fator_G={fg:.4f}')

# Deriva radiométrica
if 'entrada' in registros and 'saida' in registros:
    dR = abs(registros['entrada']['R'] - registros['saida']['R'])
    dG = abs(registros['entrada']['G'] - registros['saida']['G'])
    print(f'\n📊 Deriva: ΔR={dR:.1f}  ΔG={dG:.1f}')
    if dR > 15 or dG > 15:
        print('⚠️  Deriva significativa — condições de luz mudaram durante a coleta')
    else:
        print('✅ Condições estáveis')

# Fator final: média entrada + saída
if registros:
    FATOR_R = float(np.mean([v['fator_R'] for v in registros.values()]))
    FATOR_G = float(np.mean([v['fator_G'] for v in registros.values()]))
    FATOR_B = 1.0
    print(f'\n✅ Fator de calibração: R={FATOR_R:.4f}  G={FATOR_G:.4f}')
else:
    FATOR_R = FATOR_G = FATOR_B = 1.0
    print('⚠️  Sem painel — usando sem calibração radiométrica')

## Célula 5 — Reconstrução SfM (pycolmap)

Tempo estimado: ~5 min com A100, ~15 min com T4.

In [ ]:
import pycolmap
import shutil
import time
from pathlib import Path

IMAGE_PATH    = Path(DIRS['colmap_input'])
OUTPUT_PATH   = Path('colmap/output')
DATABASE_PATH = Path('colmap/database.db')
DENSE_PATH    = Path('colmap/dense')

# Limpeza
for p in [OUTPUT_PATH, DENSE_PATH]:
    if p.exists(): shutil.rmtree(p)
if DATABASE_PATH.exists(): DATABASE_PATH.unlink()
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
DENSE_PATH.mkdir(parents=True, exist_ok=True)

fotos = list(IMAGE_PATH.glob('*.jpg')) + list(IMAGE_PATH.glob('*.JPG'))
print(f'📷 Fotos: {len(fotos)}')

t0 = time.time()

print('\n⏳ 1/3 — Extração de features (SIFT)...')
pycolmap.extract_features(database_path=DATABASE_PATH, image_path=IMAGE_PATH)
print(f'   ✅ {(time.time()-t0)/60:.1f} min')

t1 = time.time()
print('\n⏳ 2/3 — Matching exaustivo...')
pycolmap.match_exhaustive(database_path=DATABASE_PATH)
print(f'   ✅ {(time.time()-t1)/60:.1f} min')

t2 = time.time()
print('\n⏳ 3/3 — Reconstrução esparsa (SfM)...')
maps = pycolmap.incremental_mapping(
    database_path=DATABASE_PATH,
    image_path=IMAGE_PATH,
    output_path=OUTPUT_PATH,
)
print(f'   ✅ {(time.time()-t2)/60:.1f} min')

print(f'\n📐 Modelos reconstruídos: {len(maps)}')
if not maps:
    raise RuntimeError('❌ Nenhum modelo. Verifique sobreposição das fotos (mínimo 70%).')

rec = maps[0]
n_imgs   = len(rec.images)
n_pontos = len(rec.points3D)
print(f'✅ Imagens registradas: {n_imgs} de {len(fotos)} ({n_imgs/len(fotos)*100:.0f}%)')
print(f'✅ Pontos 3D esparsos:  {n_pontos:,}')
print(f'✅ Tempo total SfM:     {(time.time()-t0)/60:.1f} min')

MODELO_PATH = str(OUTPUT_PATH / '0')
rec.write(MODELO_PATH)

# Undistortion para nuvem densa
print('\n⏳ Undistortion...')
pycolmap.undistort_images(
    output_path=DENSE_PATH,
    input_path=MODELO_PATH,
    image_path=IMAGE_PATH,
)
print('✅ Undistortion concluído')

## Célula 6 — Nuvem densa + Scale Alignment + VARI

Usa Depth Anything V2 calibrado pelos pontos esparsos SfM.  
Tempo estimado: ~2 min com A100.

In [ ]:
import torch
from transformers import pipeline
from PIL import Image
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt
import time

# Carregar modelo de profundidade
print('⏳ Carregando Depth Anything V2...')
depth_pipe = pipeline(
    task='depth-estimation',
    model='depth-anything/Depth-Anything-V2-Small-hf',
    device=0 if torch.cuda.is_available() else -1
)
print('✅ Modelo carregado')

# Carregar reconstrução SfM
rec = pycolmap.Reconstruction(MODELO_PATH)

IMAGE_PATH   = Path(DIRS['colmap_input'])
pontos_mundo = []
cores_mundo  = []
escalas      = []

print(f'\n⏳ Processando {len(rec.images)} imagens com scale alignment...')
t0 = time.time()

for img_id, img_data in rec.images.items():
    if not img_data.has_pose:
        continue
    foto_path = IMAGE_PATH / img_data.name
    if not foto_path.exists():
        continue

    img = Image.open(foto_path).convert('RGB')
    W_orig, H_orig = img.size
    img_small = img.resize((320, 240))
    depth_map = np.array(depth_pipe(img_small)['depth']).astype(float)
    H, W = depth_map.shape

    # Intrínsecos escalados
    cam = rec.cameras[img_data.camera_id]
    fx = cam.params[0] * (W / W_orig)
    fy = cam.params[0] * (H / H_orig)
    cx = cam.params[1] * (W / W_orig)
    cy = cam.params[2] * (H / H_orig)

    # Pose
    cfw = img_data.cam_from_world()
    R = cfw.rotation.matrix()
    t = cfw.translation

    # Scale alignment — calibrar depth map pelos pontos esparsos
    d_esp, d_est = [], []
    for p2d in img_data.points2D:
        if not p2d.has_point3D():
            continue
        u = p2d.xy[0] * (W / W_orig)
        v = p2d.xy[1] * (H / H_orig)
        u_i, v_i = int(round(u)), int(round(v))
        if not (0 <= u_i < W and 0 <= v_i < H):
            continue
        d_estimada = depth_map[v_i, u_i]
        if d_estimada < 0.01:
            continue
        pt3d = rec.points3D[p2d.point3D_id]
        pt_cam = R @ pt3d.xyz + t
        d_esparsa = pt_cam[2]
        if d_esparsa > 0:
            d_esp.append(d_esparsa)
            d_est.append(d_estimada)

    if len(d_esp) >= 3:
        escala = float(np.median(np.array(d_esp) / np.array(d_est)))
        escalas.append(escala)
    else:
        escala = float(np.median(escalas)) if escalas else 1.0

    depth_calibrado = depth_map * escala

    # Reprojetar para espaço mundo
    yy, xx = np.mgrid[0:H, 0:W]
    depth_flat = depth_calibrado.flatten()
    mask = depth_flat > 0.05

    X_cam = (xx.flatten()[mask] - cx) / fx * depth_flat[mask]
    Y_cam = (yy.flatten()[mask] - cy) / fy * depth_flat[mask]
    Z_cam = depth_flat[mask]

    pts_cam = np.column_stack([X_cam, Y_cam, Z_cam])
    pts_local = (R.T @ pts_cam.T).T - R.T @ t

    img_arr = np.array(img_small)
    cores = np.column_stack([
        img_arr[:,:,0].flatten()[mask],
        img_arr[:,:,1].flatten()[mask],
        img_arr[:,:,2].flatten()[mask]
    ])

    pontos_mundo.append(pts_local)
    cores_mundo.append(cores)

pontos_all = np.vstack(pontos_mundo)
cores_all  = np.vstack(cores_mundo).astype(float)
print(f'✅ {len(pontos_all):,} pontos em {(time.time()-t0)/60:.1f} min')
print(f'   Escala: min={min(escalas):.3f} max={max(escalas):.3f} mediana={np.median(escalas):.3f}')

# VARI com calibração radiométrica
R_cal = np.clip(cores_all[:,0] * FATOR_R, 0, 255)
G_cal = np.clip(cores_all[:,1] * FATOR_G, 0, 255)
B     = cores_all[:,2]
vari  = np.clip((G_cal - R_cal) / (G_cal + R_cal - B + 1e-10), -1.0, 1.0)

print(f'\n📊 VARI:')
print(f'   Média:        {vari.mean():.4f}')
print(f'   Desvio padrão: {vari.std():.4f}')
print(f'   Mínimo:       {vari.min():.4f}')
print(f'   Máximo:       {vari.max():.4f}')
print(f'   Percentil 25: {np.percentile(vari, 25):.4f}')
print(f'   Percentil 75: {np.percentile(vari, 75):.4f}')

# Filtro Statistical Outlier Removal
print('\n⏳ Filtro de outliers...')
cmap_vari = plt.get_cmap('RdYlGn')
cores_vari_full = cmap_vari((vari + 1) / 2)[:, :3]

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(pontos_all)
pcd.colors = o3d.utility.Vector3dVector(cores_vari_full)

pcd_filtrado, ind = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
vari_filtrado = vari[np.array(ind)]

removidos = len(pcd.points) - len(pcd_filtrado.points)
print(f'   Removidos: {removidos:,} ({removidos/len(pcd.points)*100:.1f}%)')
print(f'   Pontos finais: {len(pcd_filtrado.points):,}')
print('✅ Filtro aplicado')

## Célula 7 — Análise por estrato vertical

In [ ]:
import numpy as np

pontos_f = np.asarray(pcd_filtrado.points)
Z = pontos_f[:, 2]
z_min, z_max = Z.min(), Z.max()

print(f'📏 Altura reconstruída: {z_min:.2f}m – {z_max:.2f}m (total: {z_max-z_min:.2f}m)')
print(f'   Estimativa de campo: {METADATA["altura_min_estimada_m"]}m – {METADATA["altura_max_estimada_m"]}m')

N_ESTRATOS = 5
limites = np.linspace(z_min, z_max, N_ESTRATOS + 1)
estratos = []

print(f'\n📊 VARI por estrato (N={N_ESTRATOS}):')
print(f'{"Estrato":<10} {"Altura (m)":<20} {"VARI médio":<12} {"N pontos"}')
print('-' * 55)

for i in range(N_ESTRATOS):
    mask = (Z >= limites[i]) & (Z < limites[i+1])
    v_est = vari_filtrado[mask]
    if len(v_est) > 0:
        estratos.append({
            'estrato': i+1,
            'altura_min': float(limites[i]),
            'altura_max': float(limites[i+1]),
            'vari_medio': float(v_est.mean()),
            'n_pontos': int(len(v_est))
        })
        print(f'E{i+1:<9} {limites[i]:.2f}–{limites[i+1]:.2f}m{"":10} {v_est.mean():.4f}{"":6} {len(v_est):,}')

## Célula 8 — Visualizações

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

pontos_f = np.asarray(pcd_filtrado.points)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle(f'FloraCloud v0.3 — Parcela {METADATA["parcela"]} — {METADATA["data"]}', fontsize=13)

# Vista superior
sc = axes[0,0].scatter(pontos_f[:,0], pontos_f[:,1],
                       c=vari_filtrado, cmap='RdYlGn',
                       vmin=-0.3, vmax=0.3, s=0.1, alpha=0.5)
plt.colorbar(sc, ax=axes[0,0], label='VARI')
axes[0,0].set_title('Vista superior — VARI')
axes[0,0].set_aspect('equal')

# Vista lateral
axes[0,1].scatter(pontos_f[:,0], pontos_f[:,2],
                  c=vari_filtrado, cmap='RdYlGn',
                  vmin=-0.3, vmax=0.3, s=0.1, alpha=0.5)
axes[0,1].set_title('Vista lateral — VARI')
axes[0,1].set_xlabel('X (m)'); axes[0,1].set_ylabel('Z — altura (m)')

# Histograma VARI
n, bins, patches = axes[1,0].hist(vari_filtrado, bins=60, edgecolor='none')
for patch, left in zip(patches, bins[:-1]):
    patch.set_facecolor(plt.get_cmap('RdYlGn')((left + 1) / 2))
axes[1,0].axvline(vari_filtrado.mean(), color='black', linestyle='--', linewidth=1.5,
                  label=f'Média: {vari_filtrado.mean():.3f}')
axes[1,0].axvline(0, color='gray', linestyle=':', linewidth=1)
axes[1,0].set_title('Distribuição VARI')
axes[1,0].set_xlabel('VARI'); axes[1,0].legend()

# VARI por estrato
if estratos:
    varis_e = [e['vari_medio'] for e in estratos]
    labels_e = [f"E{e['estrato']}\n{e['altura_min']:.1f}–{e['altura_max']:.1f}m" for e in estratos]
    bars = axes[1,1].barh(range(len(varis_e)), varis_e,
                          color=[plt.get_cmap('RdYlGn')((v+1)/2) for v in varis_e],
                          edgecolor='gray', linewidth=0.5)
    axes[1,1].set_yticks(range(len(labels_e)))
    axes[1,1].set_yticklabels(labels_e)
    axes[1,1].axvline(0, color='gray', linewidth=0.8)
    axes[1,1].set_title('VARI médio por estrato')
    axes[1,1].set_xlabel('VARI médio')

plt.tight_layout()
fig_path = str(DIRS['resultados'] / f'floracloud_{METADATA["parcela"]}_visualizacao.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Visualização salva: {fig_path}')

## Célula 9 — Exportação e download

In [ ]:
import json
import hashlib
import zipfile
from google.colab import files
from pathlib import Path

parcela = METADATA['parcela']

# .ply com cores VARI
ply_path = str(DIRS['resultados'] / f'floracloud_{parcela}.ply')
o3d.io.write_point_cloud(ply_path, pcd_filtrado)
print(f'✅ Nuvem exportada: {ply_path} ({Path(ply_path).stat().st_size/1e6:.1f} MB)')
print('   (abrir no CloudCompare ou MeshLab para visualização 3D)')

# Relatório JSON
relatorio = {
    'floracloud_versao': '0.3',
    'metadados': METADATA,
    'sfm': {
        'imagens_registradas': len(rec.images),
        'pontos_esparsos': len(rec.points3D),
    },
    'nuvem_densa': {
        'n_pontos_bruto': int(len(pontos_all)),
        'n_pontos_filtrado': int(len(pcd_filtrado.points)),
        'escala_min': float(min(escalas)),
        'escala_max': float(max(escalas)),
        'escala_mediana': float(np.median(escalas)),
    },
    'calibracao': {
        'painel_batch': METADATA['painel_batch'],
        'fator_R': round(FATOR_R, 4),
        'fator_G': round(FATOR_G, 4),
    },
    'vari': {
        'media':        round(float(vari_filtrado.mean()), 4),
        'desvio_padrao': round(float(vari_filtrado.std()), 4),
        'minimo':       round(float(vari_filtrado.min()), 4),
        'maximo':       round(float(vari_filtrado.max()), 4),
        'p25':          round(float(np.percentile(vari_filtrado, 25)), 4),
        'p75':          round(float(np.percentile(vari_filtrado, 75)), 4),
    },
    'estratos': estratos,
    'altura_reconstruida': {
        'min_m': round(float(z_min), 3),
        'max_m': round(float(z_max), 3),
        'total_m': round(float(z_max - z_min), 3),
    }
}

json_path = str(DIRS['resultados'] / f'floracloud_{parcela}_relatorio.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(relatorio, f, ensure_ascii=False, indent=2)
print(f'✅ Relatório JSON: {json_path}')

# Downloads
print('\n📥 Baixando resultados...')
for arq in DIRS['resultados'].glob('*'):
    print(f'   {arq.name}')
    files.download(str(arq))

print('\n✅ FloraCloud v0.3 — processamento concluído!')
print(f'   Parcela:        {METADATA["parcela"]}')
print(f'   Pontos finais:  {len(pcd_filtrado.points):,}')
print(f'   VARI médio:     {vari_filtrado.mean():.4f}')